In [ ]:
# -*- coding: utf-8 -*-
"""
RF-Temp + RF-Comp treinados com 100% dos dados (treino + prova) vs Park

- RF-Temp: treinado com todas as amostras e temperaturas.
- RF-Comp: treinado com todas as amostras e temperaturas.
- Park: método físico (shift discreto + offset δS).
- Usa sempre curva real @20°C como referência.
- Mostra métricas e plots comparando Original, Ref, RF-Comp e Park.
"""

import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore", category=UserWarning)

# ========= Parâmetros =========
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"

N_BANDS       = 4
PLOT_N_EXAMPLES = 6

RF_COMP_PARAMS = dict(
    n_estimators=600,
    max_depth=None,       # ilimitado
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
)
RF_TEMP_PARAMS = dict(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    n_jobs=-1,
    random_state=7,
)

# ========= Helpers =========
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def rmsd(y_ref, y):   return float(np.sqrt(np.mean((y_ref - y)**2)))
def ccdm(y_ref, y):
    y1, y2 = y_ref - y_ref.mean(), y - y.mean()
    den = (np.linalg.norm(y1)*np.linalg.norm(y2))+1e-12
    rho = float(np.clip(np.dot(y1, y2)/den, -1, 1))
    return 1.0 - rho

def corr_per_sample(Y, Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        num=((y-y.mean())*(yh-yh.mean())).sum()
        den=np.sqrt(((y-y.mean())**2).sum()*((yh-yh.mean())**2).sum())+1e-12
        out.append(float(np.clip(num/den,-1,1)))
    return np.array(out)

def sam_per_sample(Y,Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        den=(np.linalg.norm(y)*np.linalg.norm(yh))+1e-12
        cosang=float(np.clip(np.dot(y,yh)/den,-1,1))
        out.append(float(np.degrees(np.arccos(cosang))))
    return np.array(out)

def nrmse_per_sample(Y,Yhat):
    out=[]
    for i in range(Y.shape[0]):
        y,yh=Y[i],Yhat[i]
        rmse=np.sqrt(np.mean((y-yh)**2))
        rng=np.max(y)-np.min(y)
        out.append(float(rmse/(rng+1e-12)))
    return np.array(out)

def eval_all_metrics(Y_true, Y_pred):
    y1, y2 = Y_true.reshape(-1), Y_pred.reshape(-1)
    return dict(
        R2   = r2_score(y1, y2),
        RMSE = float(np.sqrt(mean_squared_error(y1, y2))),
        MAE  = float(mean_absolute_error(y1, y2)),
        Corr = float(corr_per_sample(Y_true, Y_pred).mean()),
        SAM_deg = float(sam_per_sample(Y_true, Y_pred).mean()),
        NRMSE = float(nrmse_per_sample(Y_true, Y_pred).mean()),
        RMSD = float(np.mean([rmsd(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
        CCDM = float(np.mean([ccdm(Y_true[i], Y_pred[i]) for i in range(Y_true.shape[0])])),
    )

def print_metrics_block(title, m):
    keys = ["R2","RMSE","MAE","Corr","SAM_deg","NRMSE","RMSD","CCDM"]
    print(f"{title}: " + " | ".join([f"{k}={m[k]:.4f}" for k in keys]))

def add_extra_features(X):
    mu  = X.mean(axis=1, keepdims=True)
    sd  = X.std(axis=1,  keepdims=True)
    amp = (X.max(axis=1)-X.min(axis=1)).reshape(-1,1)
    return np.hstack([X, mu, sd, amp])

def add_temp_feature(X_aug, temp_vec):
    return np.hstack([X_aug, np.asarray(temp_vec).reshape(-1,1).astype(float)])

# ========= Park =========
def park_va_for_shift(z_ref, z, k):
    n = len(z_ref)
    if k >= 0:
        i0,i1,j0,j1 = 0, n-k, k, n
    else:
        i0,i1,j0,j1 = -k, n, 0, n+k
    if i1<=i0 or j1<=j0:
        return np.inf, 0.0, None
    zr = z_ref[i0:i1]; zd = z[j0:j1]
    delta_s = float((zr - zd).mean())
    diff = zr - (zd + delta_s)
    Va = float(np.sum(diff*diff))
    z_shift = np.full_like(z_ref, np.nan, dtype=float)
    z_shift[i0:i1] = zd + delta_s
    return Va, delta_s, z_shift

def park_compensate_curve(z_ref, z, max_shift=None):
    n = len(z_ref)
    if max_shift is None:
        max_shift = max(1, n//10)
    best = (np.inf, 0.0, None)
    for k in range(-max_shift, max_shift+1):
        Va, ds, zc = park_va_for_shift(z_ref, z, k)
        if Va < best[0]:
            best = (Va, ds, zc)
    _, _, zc = best
    if np.isnan(zc).any():
        zc = zc.copy()
        idx_valid = np.where(~np.isnan(zc))[0]
        if len(idx_valid)>0:
            first,last = idx_valid[0], idx_valid[-1]
            zc[:first] = zc[first]
            zc[last+1:] = zc[last]
        else:
            zc = z_ref.copy()
    return zc

def park_compensate_batch(y_ref, X):
    return np.vstack([park_compensate_curve(y_ref, X[i]) for i in range(X.shape[0])])

# ========= Carrega bases =========
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

# Une tudo (treino + prova)
base_all = pd.concat([base_tr, base_te], ignore_index=True)

# ========= Seleciona colunas =========
freq_cols, _ = get_freq_columns(base_all, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz = np.array([extract_freq_hz(c) for c in freq_cols], float)

# ========= Referência real @20°C =========
pool_20 = base_all.loc[base_all["temp_c"]==REF_TEMP, freq_cols].to_numpy(float)
assert len(pool_20)>0, "Não há curva real @20°C na base!"
y_ref = np.median(pool_20, axis=0)

# ========= RF-Temp (treinado com tudo) =========
X_temp = base_all[freq_cols].to_numpy(float)
X_temp_aug = add_extra_features(X_temp)
T_all = base_all["temp_c"].to_numpy(float)

rf_temp = RandomForestRegressor(**RF_TEMP_PARAMS)
rf_temp.fit(X_temp_aug, T_all)
T_hat_all = rf_temp.predict(X_temp_aug)

print("\n== RF-Temp treinado com 100% dos dados ==")
print(f"Correlação: {np.corrcoef(T_all, T_hat_all)[0,1]:.4f} | RMSE={np.sqrt(mean_squared_error(T_all, T_hat_all)):.4f} | MAE={mean_absolute_error(T_all, T_hat_all):.4f}")

# ========= RF-Comp (treinado com tudo) =========
X_all = base_all[freq_cols].to_numpy(float)
Y_target = (y_ref[None, :] - X_all)
X_aug = add_extra_features(X_all)
X_comp = add_temp_feature(X_aug, T_all)

t0=time.time()
rf_comp = RandomForestRegressor(**RF_COMP_PARAMS)
rf_comp.fit(X_comp, Y_target)
print(f"[INFO] Tempo RF-Comp (treino total): {time.time()-t0:.1f}s")

Y_hat = X_all + rf_comp.predict(X_comp)
Y_ref = np.tile(y_ref, (Y_hat.shape[0], 1))
Y_hat_park = park_compensate_batch(y_ref, X_all)

# ========= Métricas =========
print("\n== MÉTRICAS (treino total)==")
print_metrics_block("RF-Comp (tudo)", eval_all_metrics(Y_ref,  Y_hat))
print_metrics_block("Park    (tudo)", eval_all_metrics(Y_ref,  Y_hat_park))

# ========= Métricas por banda =========
def band_indices(n, n_bands): return [np.array(ix,int) for ix in np.array_split(np.arange(n), n_bands)]
idx_bands = band_indices(Y_ref.shape[1], N_BANDS)
print("\n== MÉTRICAS por banda ==")
for b, ix in enumerate(idx_bands):
    mr = eval_all_metrics(Y_ref[:,ix], Y_hat[:,ix])
    mp = eval_all_metrics(Y_ref[:,ix], Y_hat_park[:,ix])
    print(f"Banda {b}: RF-Comp -> R2={mr['R2']:.3f} | RMSE={mr['RMSE']:.3f} | MAE={mr['MAE']:.3f} | RMSD={mr['RMSD']:.3f} | CCDM={mr['CCDM']:.3f}")
    print(f"          Park    -> R2={mp['R2']:.3f} | RMSE={mp['RMSE']:.3f} | MAE={mp['MAE']:.3f} | RMSD={mp['RMSD']:.3f} | CCDM={mp['CCDM']:.3f}")

# ========= Plots =========
def pick_examples(base_df, n=PLOT_N_EXAMPLES):
    temps = sorted(base_df["temp_c"].unique())
    idxs=[]
    for T in temps:
        cand = np.where(base_df["temp_c"].to_numpy(float)==T)[0]
        if len(cand): idxs.append(int(cand[0]))
    rest = [i for i in range(len(base_df)) if i not in idxs]
    idxs += rest[:max(0, n-len(idxs))]
    return idxs[:n]

def plot_examples(fhz, y_ref, X_orig, Y_rf, Y_pk, base_df, n=PLOT_N_EXAMPLES):
    fhz_khz = fhz/1e3
    idxs = pick_examples(base_df, n)
    for i in idxs:
        plt.figure(figsize=(8,4.5))
        plt.plot(fhz_khz, X_orig[i], label=f"Original @ {base_df.iloc[i]['temp_c']}°C", lw=1.2)
        plt.plot(fhz_khz, y_ref,     label=f"Referência @ {REF_TEMP}°C", lw=1.8)
        plt.plot(fhz_khz, Y_rf[i],   label="RF-Comp → 20°C", lw=1.5)
        plt.plot(fhz_khz, Y_pk[i],   label="Park → 20°C", lw=1.5)
        plt.xlabel("Frequência (kHz)"); plt.ylabel("Impedância (un.)")
        plt.title(f"Amostra {i} — Compensação para {REF_TEMP}°C")
        plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
    plt.show()

print("\n== PLOTS de exemplos ==")
plot_examples(fhz, y_ref, X_all, Y_hat, Y_hat_park, base_all, n=PLOT_N_EXAMPLES)

print("\n[FIM]")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_rfcomp_e_park(base_df, Y_rf, Y_park, y_ref, fhz, temp_escolhida,
                       offset_rf=0.015, offset_pk=0.020):
    """
    Plota RF-Comp + Ref e Park + Ref em dois gráficos separados,
    com offsets diferenciados e contraste otimizado.
    """

    # ===== Aparência global =====
    plt.rcParams.update({
        'font.size': 20,
        'text.usetex': False,
        'font.family': "Times New Roman"
    })

    fhz_khz = fhz / 1e3
    mask = np.isclose(base_df["temp_c"].to_numpy(float), temp_escolhida)
    idxs = np.where(mask)[0]

    if not len(idxs):
        print(f"Nenhuma curva encontrada em {temp_escolhida}°C.")
        return

    i = idxs[0]  # primeira curva dessa temperatura

    # deslocamentos para evitar sobreposição direta
    y_ref_rf = y_ref + offset_rf * np.mean(y_ref)
    y_ref_pk = y_ref + offset_pk * np.mean(y_ref)

    # ==========================================================
    # -------------------- RF-Comp --------------------
    fig, ax = plt.subplots(figsize=(6.2,5))
    ax.plot(fhz_khz, Y_rf[i], '-', lw=2.0, alpha=0.9, color='royalblue', label='RF-Comp 20°C')
    ax.plot(fhz_khz, y_ref_rf, '--', lw=0.8, color='mediumpurple', alpha=0.95, label='Referência 20°C')

    ax.set_xlabel('Frequência [kHz]', fontsize=20, labelpad=20)
    ax.set_ylabel('Real [Ω]', fontsize=20, rotation=90, labelpad=20)
    ax.set_xlim([fhz_khz[0], fhz_khz[-1]])
    ax.set_ylim([
        min(np.min(y_ref_rf), np.min(Y_rf[i])) * 0.97,
        max(np.max(y_ref_rf), np.max(Y_rf[i])) * 1.03
    ])
    ax.grid(True, alpha=0.3)
    leg = ax.legend(fontsize=14, loc='upper right', frameon=True, edgecolor='0.7')
    leg.get_frame().set_alpha(0.8)
    ax.set_title(f'Compensação RF-Comp — {temp_escolhida}°C', fontsize=20)
    fig.tight_layout()
    plt.show()

    # ==========================================================
    # -------------------- Park --------------------
    fig, ax = plt.subplots(figsize=(6.2,5))
    ax.plot(fhz_khz, Y_park[i], '-', lw=2.0, alpha=0.9, color='royalblue', label='Park 20°C')
    ax.plot(fhz_khz, y_ref_pk, '--', lw=0.8, color='mediumpurple', alpha=0.95, label='Referência 20°C')

    ax.set_xlabel('Frequência [kHz]', fontsize=20, labelpad=20)
    ax.set_ylabel('Real[Ω]', fontsize=20, rotation=90, labelpad=20)
    ax.set_xlim([fhz_khz[0], fhz_khz[-1]])
    ax.set_ylim([
        min(np.min(y_ref_pk), np.min(Y_park[i])) * 0.97,
        max(np.max(y_ref_pk), np.max(Y_park[i])) * 1.03
    ])
    ax.grid(True, alpha=0.3)
    leg = ax.legend(fontsize=14, loc='upper right', frameon=True, edgecolor='0.7')
    leg.get_frame().set_alpha(0.8)
    ax.set_title(f'Compensação Park — {temp_escolhida}°C', fontsize=20)
    fig.tight_layout()
    plt.show()


# ==========================================================
# 🧩 EXEMPLO DE USO
# ==========================================================
# base_all  → DataFrame completo
# Y_hat     → curvas compensadas pelo RF-Comp
# Y_hat_park→ curvas compensadas pelo Park
# y_ref     → curva referência (20°C)
# fhz       → vetor de frequências em Hz

# Exemplo de chamada:
plot_rfcomp_e_park(base_all, Y_hat, Y_hat_park, y_ref, fhz, temp_escolhida=40)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==========================================================
# Função de métricas
# ==========================================================
def curva_metrics(y_ref, y_orig, y_comp):
    """Calcula métricas de forma e adaptação térmica."""
    y_ref_n  = (y_ref - np.mean(y_ref)) / np.std(y_ref)
    y_comp_n = (y_comp - np.mean(y_comp)) / np.std(y_comp)
    y_orig_n = (y_orig - np.mean(y_orig)) / np.std(y_orig)

    corr = np.corrcoef(y_ref_n, y_comp_n)[0, 1]
    cosang = np.dot(y_ref_n, y_comp_n) / (np.linalg.norm(y_ref_n) * np.linalg.norm(y_comp_n))
    sam = np.degrees(np.arccos(np.clip(cosang, -1, 1)))
    rmsd = np.sqrt(np.mean((y_ref - y_comp) ** 2))
    E_res = np.sum((y_ref - y_comp) ** 2) / np.sum((y_ref - y_orig) ** 2)

    return {
        "Corr": corr,
        "SAM_deg": sam,
        "RMSD": rmsd,
        "Energy_Ratio": E_res,
        "Adaptation_Index": 1 - E_res
    }

# ==========================================================
# Função principal de plotagem + métricas
# ==========================================================
def plot_rfcomp_e_park(base_df, Y_rf, Y_park, y_ref, fhz, temp_escolhida,
                       offset_rf=0.015, offset_pk=0.020):
    """
    Plota RF-Comp + Ref e Park + Ref em dois gráficos separados
    e imprime métricas de forma/adaptação para cada método.
    """

    plt.rcParams.update({
        'font.size': 20,
        'text.usetex': False,
        'font.family': "Times New Roman"
    })

    fhz_khz = fhz / 1e3
    mask = np.isclose(base_df["temp_c"].to_numpy(float), temp_escolhida)
    idxs = np.where(mask)[0]

    # Verifica se há alguma curva na temperatura escolhida
    if not len(idxs):
        print(f"Nenhuma curva encontrada em {temp_escolhida}°C.")
        return

    i = idxs[0]  # primeira curva dessa temperatura

    # Seleciona somente as colunas de frequência que correspondem a fhz
    freq_cols = [c for c in base_df.columns if c.startswith("f_")]
    y_orig = base_df.loc[i, freq_cols].to_numpy(float)
    if len(y_orig) != len(y_ref):
        y_orig = y_orig[:len(y_ref)]  # recorta se precisar

    # Cria cópias levemente deslocadas para contraste visual
    y_ref_rf = y_ref + offset_rf * np.mean(y_ref)
    y_ref_pk = y_ref + offset_pk * np.mean(y_ref)

    # ======================================================
    # --- RF-Comp ---
    m_rf = curva_metrics(y_ref, y_orig, Y_rf[i])

    fig, ax = plt.subplots(figsize=(6.2, 5))
    ax.plot(fhz_khz, Y_rf[i], '-', lw=2.0, alpha=0.9, color='royalblue', label='RF-Comp 20°C')
    ax.plot(fhz_khz, y_ref_rf, '--', lw=1.2, color='black', alpha=0.9, label='Referência 20°C')

    ax.set_xlabel('Frequência [kHz]', fontsize=20, labelpad=20)
    ax.set_ylabel('Impedância [un.]', fontsize=20, rotation=90, labelpad=20)
    ax.set_xlim([fhz_khz[0], fhz_khz[-1]])
    ax.grid(True, alpha=0.3)
    leg = ax.legend(fontsize=14, loc='upper right', frameon=True, edgecolor='0.7')
    leg.get_frame().set_alpha(0.8)
    ax.set_title(f'Compensação RF-Comp — {temp_escolhida}°C', fontsize=20)
    fig.tight_layout()
    plt.show()

    print(f"\n=== MÉTRICAS RF-Comp ({temp_escolhida}°C) ===")
    for k, v in m_rf.items():
        print(f"{k:<15}: {v:.4f}")

    # ======================================================
    # --- Park ---
    m_pk = curva_metrics(y_ref, y_orig, Y_park[i])

    fig, ax = plt.subplots(figsize=(6.2, 5))
    ax.plot(fhz_khz, Y_park[i], '-', lw=2.0, alpha=0.9, color='crimson', label='Park 20°C')
    ax.plot(fhz_khz, y_ref_pk, '--', lw=1.2, color='black', alpha=0.9, label='Referência 20°C')

    ax.set_xlabel('Frequência [kHz]', fontsize=20, labelpad=20)
    ax.set_ylabel('Impedância [un.]', fontsize=20, rotation=90, labelpad=20)
    ax.set_xlim([fhz_khz[0], fhz_khz[-1]])
    ax.grid(True, alpha=0.3)
    leg = ax.legend(fontsize=14, loc='upper right', frameon=True, edgecolor='0.7')
    leg.get_frame().set_alpha(0.8)
    ax.set_title(f'Compensação Park — {temp_escolhida}°C', fontsize=20)
    fig.tight_layout()
    plt.show()

    print(f"\n=== MÉTRICAS Park ({temp_escolhida}°C) ===")
    for k, v in m_pk.items():
        print(f"{k:<15}: {v:.4f}")


# ==========================================================
# 🧩 Exemplo de uso
# ==========================================================
# plot_rfcomp_e_park(base_all, Y_hat, Y_hat_park, y_ref, fhz, temp_escolhida=40)

In [ ]:
plot_rfcomp_e_park(base_all, Y_hat, Y_hat_park, y_ref, fhz, temp_escolhida=40)

In [ ]:
import numpy as np

def curva_originalidade(y_orig, y_comp):
    """
    Mede o quanto a curva compensada mantém a forma da curva original.
    Retorna métricas de similaridade e um índice de originalidade agregado.
    """
    # Normalização (remove offset e escala)
    y_orig_n = (y_orig - np.mean(y_orig)) / np.std(y_orig)
    y_comp_n = (y_comp - np.mean(y_comp)) / np.std(y_comp)

    # Correlação linear
    corr = np.corrcoef(y_orig_n, y_comp_n)[0, 1]

    # SAM (Spectral Angle Mapper)
    cosang = np.dot(y_orig_n, y_comp_n) / (np.linalg.norm(y_orig_n) * np.linalg.norm(y_comp_n))
    sam = np.degrees(np.arccos(np.clip(cosang, -1, 1)))

    # RMSD (diferença média absoluta)
    rmsd = np.sqrt(np.mean((y_orig - y_comp) ** 2))

    # Índice composto de originalidade
    originality = corr * (1 - sam / 90)  # varia de 0 (nenhuma) a 1 (idêntica)

    return {
        "Corr": corr,
        "SAM_deg": sam,
        "RMSD": rmsd,
        "Originality_Index": originality
    }

# ==========================================================
# 🧩 Exemplo de uso direto com as variáveis já existentes
# ==========================================================

def avaliar_originalidade(base_df, Y_rf, Y_park, fhz, temp_escolhida):
    """
    Calcula o quanto as curvas compensadas (RF e Park)
    preservam a forma da curva original.
    """

    # Seleciona curva original (sem compensação)
    mask = np.isclose(base_df["temp_c"].to_numpy(float), temp_escolhida)
    idxs = np.where(mask)[0]
    if not len(idxs):
        print(f"Nenhuma curva encontrada em {temp_escolhida}°C.")
        return
    i = idxs[0]

    # Extrai a curva original (mesmo comprimento das curvas compensadas)
    freq_cols = [c for c in base_df.columns if c.startswith("f_")]
    y_orig = base_df.loc[i, freq_cols].to_numpy(float)
    if len(y_orig) > Y_rf.shape[1]:
        y_orig = y_orig[:Y_rf.shape[1]]  # corta se precisar
    elif len(y_orig) < Y_rf.shape[1]:
        y_orig = np.pad(y_orig, (0, Y_rf.shape[1]-len(y_orig)), mode='edge')

    # Obtém as curvas compensadas correspondentes
    y_comp_rf = Y_rf[i]
    y_comp_pk = Y_park[i]

    # Calcula métricas de originalidade
    m_rf = curva_originalidade(y_orig, y_comp_rf)
    m_pk = curva_originalidade(y_orig, y_comp_pk)

    # Exibe resultados
    print(f"\n=== ORIGINALIDADE ({temp_escolhida}°C) ===")
    print("→ RF-Comp:")
    for k, v in m_rf.items():
        print(f"{k:<15}: {v:.4f}")
    print("\n→ Park:")
    for k, v in m_pk.items():
        print(f"{k:<15}: {v:.4f}")

    return m_rf, m_pk


# ==========================================================
# 🔧 CHAMADA DE EXEMPLO
# ==========================================================
# Basta rodar isso depois de já ter suas variáveis carregadas:
# base_all, Y_hat, Y_hat_park, fhz

avaliar_originalidade(base_all, Y_hat, Y_hat_park, fhz, temp_escolhida=40)
